In [1]:
# packages
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import json

In [2]:
# set model
model = "bert-base"

# set layer
layer = "mean"

# set column name?

# set file
sample = pd.read_parquet("/kaggle/input/datasets/lianestrauch/bert-base-samples/sample_bert_base_mean_last4.parquet")

# set eps
eps_values = {
    5: np.round(np.arange(0.04, 0.16 + 0.01, 0.01), 2),
    10: np.round(np.arange(0.05, 0.16 + 0.01, 0.01), 2),
    50: np.round(np.arange(0.07, 0.16 + 0.01, 0.01), 2),
    100: np.round(np.arange(0.08, 0.16 + 0.01, 0.01), 2),
}

# set outputfile
output_file = Path(f"clustering_results_{model}_{layer}.json")


In [3]:
SAMPLE_PATH = Path("/kaggle/input/datasets/lianestrauch/bert-base-samples")

In [4]:
!pip install kDBCV

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 41.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 r

In [5]:
import json
import time
from pathlib import Path
import numpy as np

# Patch NumPy 2.0 compatibility for legacy libraries like kDBCV
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'int_'):
    np.int_ = np.int64

from kDBCV import DBCV_score
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize

from sklearn.cluster import DBSCAN

# ============================================================
# Load existing results, or create a new dictionary
# ============================================================

if output_file.exists():
    with open(output_file, "r") as f:
        clustering_results = json.load(f)

    print(
        f"Loaded {len(clustering_results)} existing clustering results."
    )
else:
    clustering_results = {}

    print("No existing results found. Starting a new file.")


# ============================================================
# Prepare embeddings
# ============================================================

col_name = sample.columns[-1]

embeddings = np.vstack(sample[col_name].values)
n_samples = len(embeddings)

print(f"Number of data points: {n_samples}")


# ============================================================
# Run clustering
# ============================================================

for min_samples, current_eps_values in eps_values.items():

    for eps in current_eps_values:

        key = f"eps_{eps:.4f}_minPts_{min_samples}"

        # ----------------------------------------------------
        # Skip if this combination has already been calculated
        # ----------------------------------------------------
        if key in clustering_results:
            print(
                f"SKIPPING: eps={eps:.4f}, "
                f"minPts={min_samples} "
                f"(already calculated)"
            )
            continue

        print(
            f"Running: eps={eps:.4f}, "
            f"minPts={min_samples}..."
        )

        start_time = time.perf_counter()

        labels = DBSCAN(
            eps=eps,
            min_samples=min_samples,
            metric="cosine"
        ).fit_predict(embeddings)

        elapsed_time = time.perf_counter() - start_time

        # ----------------------------------------------------
        # Cluster statistics
        # ----------------------------------------------------

        unique_labels, counts = np.unique(
            labels,
            return_counts=True
        )

        # Exclude noise (-1)
        cluster_counts = counts[unique_labels != -1]

        n_clusters = len(cluster_counts)
        n_noise = int(np.sum(labels == -1))

        # Largest cluster
        if len(cluster_counts) > 0:
            largest_cluster_size = int(np.max(cluster_counts))
        else:
            largest_cluster_size = 0

        # Percentage of full dataset
        largest_cluster_pct = (
            100 * largest_cluster_size / n_samples
            if n_samples > 0 else 0
        )

        # DBCV
        embeddings_norm = normalize(embeddings, norm='l2', axis=1) # dbcv only takes euclidean distance
        score = DBCV_score(embeddings_norm, labels)
        print("DBCV Score (based on normalised embeddings and euclidean distance):", score)

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        clustering_results[key] = {
            "model": model,
            "layer": layer,
            "eps": float(eps),
            "min_samples": int(min_samples),
            "n_clusters": int(n_clusters),
            "n_noise": n_noise,
            "largest_cluster_size": largest_cluster_size,
            "largest_cluster_pct": float(largest_cluster_pct),
            "runtime_seconds": float(elapsed_time),
            "n_samples": int(n_samples),
            "dbcv": score,
            "labels": { str(sample_id): int(label) for sample_id, label in zip(sample["id"], labels) }
        }

        print(
            f"  finished: "
            f"{n_clusters} clusters, "
            f"largest={largest_cluster_size} "
            f"({largest_cluster_pct:.2f}%), "
            f"time={elapsed_time:.2f}s"
        )

        # ----------------------------------------------------
        # Save immediately after each clustering
        # ----------------------------------------------------
        #
        # This is useful for long-running experiments:
        # if the script crashes halfway through, everything
        # completed so far is already saved.
        #

        with open(output_file, "w") as f:
            json.dump(clustering_results, f)


# ============================================================
# Done
# ============================================================

print(
    f"\nDone. Total stored results: "
    f"{len(clustering_results)}"
)

No existing results found. Starting a new file.
Number of data points: 49919
Running: eps=0.0400, minPts=5...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.0010289120178222648), None)
  finished: 14 clusters, largest=29 (0.06%), time=75.76s
Running: eps=0.0500, minPts=5...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.0017123845764861704), None)
  finished: 26 clusters, largest=86 (0.17%), time=75.00s
Running: eps=0.0600, minPts=5...


...

DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(-6.259711265289486e-05), None)
  finished: 4 clusters, largest=2727 (5.46%), time=73.16s
Running: eps=0.1100, minPts=100...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(-0.028298402830978314), None)
  finished: 4 clusters, largest=6091 (12.20%), time=73.16s
Running: eps=0.1200, minPts=100...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(-0.036509536811020295), None)
  finished: 4 clusters, largest=11069 (22.17%), time=73.10s
Running: eps=0.1300, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 3 clusters, largest=27691 (55.47%), time=73.37s
Running: eps=0.1400, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 2 clusters, largest=35983 (72.08%), time=73.41s
Running: eps=0.1500, minPts=100...
Not enough clusters: must have at least two.
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 1 clusters, largest=41712 (83.56%), time=73.49s
Running: eps=0.1600, minPts=100...
Not enough clusters: must have at least two.
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 1 clusters, largest=45418 (90.98%), time=73.91s
